cuda


In [ ]:
import os
import cv2
import torch
import numpy as np
from tqdm import tqdm

from concurrent.futures import ThreadPoolExecutor
from torchvision.models.video import r3d_18, R3D_18_Weights

# PATHS
INPUT_DIR = '/kaggle/input/datasets/odins0n/ucf-crime-dataset/Train'
OUTPUT_DIR = '/kaggle/working/extracted_features'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# DEVICE
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# MODEL
weights = R3D_18_Weights.DEFAULT
model = r3d_18(weights=weights)
model.fc = torch.nn.Identity()
model = model.to(device).eval()

# NORMALIZATION (SAFE FLOAT32)
mean = torch.tensor(
    [0.43216, 0.394666, 0.37645],
    dtype=torch.float32,
    device=device
).view(1, 3, 1, 1, 1)

std = torch.tensor(
    [0.22803, 0.22145, 0.216989],
    dtype=torch.float32,
    device=device
).view(1, 3, 1, 1, 1)

# GROUP FRAMES
def group_frames(folder):
    videos = {}

    for file in os.listdir(folder):
        if not file.endswith('.png'):
            continue

        key = "_".join(file.split("_")[:2])

        if key not in videos:
            videos[key] = []

        videos[key].append(file)

    for k in videos:
        videos[k].sort(key=lambda x: int(x.split("_")[-1].split(".")[0]))

    return videos

# PARALLEL FRAME LOADING
def load_frame(path):
    img = cv2.imread(path)
    if img is not None:
        return cv2.resize(img, (112, 112))
    return None

# FEATURE EXTRACTION (GPU OPTIMIZED)
def extract_features(frames, segments=32, clip_len=16):
    if len(frames) == 0:
        return None

    # pad short videos
    if len(frames) < clip_len:
        while len(frames) < clip_len:
            frames.append(frames[-1])

    indices = np.linspace(0, len(frames) - clip_len, segments).astype(int)

    clips = []
    for idx in indices:
        clip = frames[idx:idx + clip_len]

        clip = np.array(clip)
        clip = torch.from_numpy(clip).permute(3, 0, 1, 2).float() / 255.0
        clips.append(clip)

    # 🔥 BIG GPU BATCH
    clips = torch.stack(clips).to(device).float()

    # normalize (safe on GPU now)
    clips = (clips - mean) / std

    # 🔥 SINGLE FORWARD PASS
    with torch.no_grad():
        features = model(clips)

    return features.cpu().numpy()

# MAIN LOOP
total_saved = 0

for category in os.listdir(INPUT_DIR):
    cat_path = os.path.join(INPUT_DIR, category)

    if not os.path.isdir(cat_path):
        continue

    print(f"\nProcessing: {category}")

    videos = group_frames(cat_path)
    print(f"Found {len(videos)} videos")

    save_dir = os.path.join(OUTPUT_DIR, category)
    os.makedirs(save_dir, exist_ok=True)

    for vid_name, frame_files in tqdm(videos.items()):

        # 🔥 PARALLEL FRAME LOADING
        with ThreadPoolExecutor(max_workers=8) as executor:
            frames = list(executor.map(
                lambda f: load_frame(os.path.join(cat_path, f)),
                frame_files
            ))

        frames = [f for f in frames if f is not None]

        feat = extract_features(frames)

        if feat is not None:
            np.save(os.path.join(save_dir, vid_name + '.npy'), feat)
            total_saved += 1

print("\nSaved features:", total_saved)

# VERIFY OUTPUT
import glob
files = glob.glob('/kaggle/working/extracted_features/**/*.npy', recursive=True)

print("Total feature files:", len(files))

if len(files) > 0:
    sample = np.load(files[0])

    print("Sample shape:", sample.shape)  # should be (32, 512)